# Fine-tuning NLLB-200 3.3B — bribri ↔ español

Notebook del **Avance 5** para ejecutar en Google Colab Pro+ con GPU A100.

Objetivo: repetir el protocolo experimental del NLLB previo con una red más grande (`facebook/nllb-200-3.3B`), conservando semilla, splits, métrica principal (`chrF/chrF++`) y batch efectivo comparable. La notebook escribe checkpoints reanudables en Google Drive para sobrevivir desconexiones o reinicios del runtime.

Configuración esperada de Colab:

1. Runtime → Change runtime type → GPU.
2. Seleccionar A100 cuando esté disponible.
3. Ejecutar las celdas en orden.
4. Si el runtime se cae, volver a ejecutar desde el inicio: `train()` detecta `checkpoint-last` y reanuda.

## 1. Montar Drive y clonar repo

Drive se monta antes del entrenamiento porque los checkpoints de 3.3B son grandes y no deben depender del disco efímero de `/content`.

In [ ]:
import os
import pathlib
import subprocess
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('No se montó Drive automáticamente:', exc)

REPO_URL = 'https://github.com/dleiva98/Proyecto-Integrador.git'
BRANCH = 'avance-5'
REPO_DIR = pathlib.Path('/content/proyecto-integrador')
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/proyecto-integrador-avance5')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)
# Colab conserva /content entre ejecuciones del notebook. Forzamos el checkout
# remoto para evitar importar una versión vieja de nllb_train.py.
subprocess.check_call(['git', 'fetch', 'origin', BRANCH])
subprocess.check_call(['git', 'checkout', '-B', BRANCH, f'origin/{BRANCH}'])

commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
log_line = subprocess.check_output(['git', 'log', '-1', '--oneline'], text=True).strip()
print('cwd:', os.getcwd())
print('commit:', log_line)
print('drive output root:', DRIVE_ROOT)


## 2. Dependencias y verificación de hardware

Para 3.3B hay dos rutas: full fine-tuning en A100 >=39 GiB, o QLoRA 4-bit en L4 >=20 GiB con RAM amplia >=40 GiB. Si Colab asigna otra GPU, la notebook aborta antes de cargar el modelo.

In [ ]:
%pip install -q torch "transformers==4.48.3" accelerate sentencepiece bitsandbytes peft tqdm sacrebleu pydantic polars pyarrow matplotlib psutil

import psutil
import torch

FULL_A100_MIN_VRAM_GB = 39.0
L4_MIN_VRAM_GB = 20.0
L4_MIN_SYSTEM_RAM_GB = 40.0

print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Esta notebook requiere GPU CUDA. En Colab: Runtime > Change runtime type > GPU.')

gpu_name = torch.cuda.get_device_name(0)
free, total = torch.cuda.mem_get_info()
total_gb = total / 1024**3
free_gb = free / 1024**3
system_ram_gb = psutil.virtual_memory().total / 1024**3
print('gpu :', gpu_name)
print(f'vram: {free_gb:.1f} GiB free / {total_gb:.1f} GiB total')
print(f'ram : {system_ram_gb:.1f} GiB total')

if 'A100' in gpu_name and total_gb >= FULL_A100_MIN_VRAM_GB:
    TRAINING_MODE = 'full_a100'
elif 'L4' in gpu_name and total_gb >= L4_MIN_VRAM_GB and system_ram_gb >= L4_MIN_SYSTEM_RAM_GB:
    TRAINING_MODE = 'qlora_l4'
else:
    raise RuntimeError(
        'Recursos insuficientes para NLLB 3.3B. Se requiere A100 >=39 GiB para full fine-tuning '
        f'o L4 >=20 GiB con RAM amplia >=40 GiB para QLoRA. Colab asignó {gpu_name}, '
        f'VRAM={total_gb:.1f} GiB, RAM={system_ram_gb:.1f} GiB.'
    )

print('training mode:', TRAINING_MODE)


## 2.1 Limpieza preventiva de memoria

Si una corrida previa falló por OOM, ejecuta esta celda o reinicia el runtime antes de volver a entrenar.


In [ ]:
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
print('memoria CUDA liberada')


## 3. Splits reproducibles

Se reutilizan `data/splits/{train,val,test}.jsonl`. Si faltan, se regeneran con `scripts/make_splits.py`, que usa semilla 42 y estratificación por `(domain, confidence)`.

In [ ]:
from pathlib import Path

splits = Path('data/splits')
if not (splits / 'train.jsonl').exists():
    subprocess.check_call([sys.executable, 'scripts/make_splits.py'])
else:
    print('splits ya existen')

for split_name in ['train', 'val', 'test']:
    path = splits / f'{split_name}.jsonl'
    print(split_name, sum(1 for _ in path.open()), 'pares')

## 4. Heartbeat de Colab

Esto no usa hacks de navegador ni evita políticas de Colab. Sólo mantiene salida periódica con hora, RAM y VRAM para que el entrenamiento no parezca inactivo y para facilitar diagnóstico si el runtime se corta.

In [ ]:
import datetime as dt
import threading
import time
import psutil
import torch

_HEARTBEAT_STOP = False

def heartbeat(interval_seconds=300):
    while not _HEARTBEAT_STOP:
        parts = [f"heartbeat {dt.datetime.now().isoformat(timespec='seconds')}"]
        parts.append(f"ram_used={psutil.virtual_memory().percent:.1f}%")
        if torch.cuda.is_available():
            free, total = torch.cuda.mem_get_info()
            parts.append(f"vram_free={free/1024**3:.1f}/{total/1024**3:.1f}GiB")
        print(' | '.join(parts), flush=True)
        time.sleep(interval_seconds)

threading.Thread(target=heartbeat, daemon=True).start()

## 5. Entrenamiento 3.3B con checkpoints reanudables (A100 full o L4 QLoRA)

Parámetros comparables con la corrida base:

| Parámetro | Valor |
|---|---:|
| Modelo | `facebook/nllb-200-3.3B` |
| Épocas | 3 |
| Learning rate | `5e-4` |
| Batch efectivo | 8 |
| Micro-batch físico | 1 |
| Acumulación de gradiente | 8 |
| Precisión | bf16 |
| Optimizador | AdamW 8-bit |
| Checkpoint | cada 100 pasos de optimizador |
| Selección de mejor checkpoint | `chrF++` promedio de validación |

Si la sesión se desconecta, vuelve a ejecutar las celdas desde el inicio. Con `resume_if_checkpoint_exists=True`, el entrenamiento carga `outputs_nllb_3_3b/checkpoint-last` desde Drive.

In [ ]:
import importlib
import sys
from dataclasses import fields
from pathlib import Path

sys.path.insert(0, str(Path('src').resolve()))
import voces_corpus.training.nllb_train as nllb_train
importlib.reload(nllb_train)

TrainConfig = nllb_train.TrainConfig
train = nllb_train.train

required_fields = {'load_in_4bit', 'lora_r', 'min_system_memory_gb'}
available_fields = {field.name for field in fields(TrainConfig)}
missing_fields = required_fields - available_fields
if missing_fields:
    raise RuntimeError(
        f'TrainConfig está desactualizado; faltan campos {sorted(missing_fields)}. '
        'Ejecuta de nuevo la celda 1 de setup para sincronizar origin/avance-5, '
        'o reinicia el runtime y corre la notebook desde el inicio.'
    )

MODEL_NAME = 'facebook/nllb-200-3.3B'
OUTPUT_DIR = DRIVE_ROOT / f'outputs_nllb_3_3b_{TRAINING_MODE}'

common_kwargs = dict(
    model_str=MODEL_NAME,
    tokenizer_str=MODEL_NAME,
    epochs=3,
    batch_size=1,
    gradient_accumulation_steps=8,
    lr=5e-4,
    eval_every=100,
    checkpoint_every=100,
    log_every=10,
    use_float16=False,
    use_bfloat16=True,
    save_model_on_evaluation=True,
    best_metric='chrfpp',
    resume_if_checkpoint_exists=True,
    output_dir=OUTPUT_DIR,
)

if TRAINING_MODE == 'full_a100':
    cfg = TrainConfig(
        **common_kwargs,
        optimizer_name='adamw_bnb_8bit',
        required_gpu_name='A100',
        min_cuda_memory_gb=39.0,
    )
elif TRAINING_MODE == 'qlora_l4':
    cfg = TrainConfig(
        **common_kwargs,
        load_in_4bit=True,
        lora_r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        optimizer_name='adamw_torch',
        required_gpu_name='L4',
        min_cuda_memory_gb=20.0,
        min_system_memory_gb=40.0,
    )
else:
    raise ValueError(f'TRAINING_MODE no soportado: {TRAINING_MODE}')

history = train(cfg, repo_root=Path.cwd())
print('Entrenamiento terminado. Outputs:', OUTPUT_DIR)


## 6. Evaluación final sobre TEST

Se evalúa el mejor checkpoint por `chrF++` de validación. Si no existe, se usa `final_nllb`.

In [ ]:
import json
from torch.utils.data import DataLoader
from transformers import DataCollatorForSeq2Seq
from voces_corpus.training.nllb_train import TranslationDataset, evaluate_split, _read_jsonl, load_trained_model


def select_model_dir(output_dir: Path) -> Path:
    candidates = []
    for path in output_dir.glob('best_nllb_chrfpp=*'):
        try:
            score = float(path.name.split('=', 1)[1])
        except ValueError:
            continue
        candidates.append((score, path))
    if candidates:
        return max(candidates, key=lambda item: item[0])[1]
    return output_dir / 'final_nllb'

MODEL_DIR = select_model_dir(OUTPUT_DIR)
print('Evaluando checkpoint:', MODEL_DIR)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, tokenizer = load_trained_model(MODEL_DIR, cfg, device)

if hasattr(model.config, 'use_cache'):
    model.config.use_cache = False

test_data = _read_jsonl(Path('data/splits/test.jsonl'))
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding='longest', max_length=cfg.max_length, pad_to_multiple_of=8)

def make_test_loader(src_lang, tgt_lang, src_tok, tgt_tok):
    ds = TranslationDataset(test_data, src_lang, tgt_lang, src_tok, tgt_tok, False, tokenizer, cfg.max_length)
    return DataLoader(ds, batch_size=1, shuffle=False, collate_fn=collator)

test_s2t = make_test_loader(cfg.src_lang, cfg.tgt_lang, cfg.src_lang_token, cfg.tgt_lang_token)
test_t2s = make_test_loader(cfg.tgt_lang, cfg.src_lang, cfg.tgt_lang_token, cfg.src_lang_token)

res_s2t = evaluate_split(model, tokenizer, test_s2t, cfg.tgt_lang_token, device, cfg.max_length)
res_t2s = evaluate_split(model, tokenizer, test_t2s, cfg.src_lang_token, device, cfg.max_length)

summary = {
    'model': cfg.model_str,
    'run': f'nllb_3_3b_colab_{TRAINING_MODE}',
    'checkpoint': str(MODEL_DIR),
    'es->bri': {k: res_s2t[k] for k in ('eval_loss', 'spbleu', 'chrf', 'chrfpp')},
    'bri->es': {k: res_t2s[k] for k in ('eval_loss', 'spbleu', 'chrf', 'chrfpp')},
    'avg': {
        'eval_loss': (res_s2t['eval_loss'] + res_t2s['eval_loss']) / 2,
        'spbleu':    (res_s2t['spbleu']    + res_t2s['spbleu'])    / 2,
        'chrf':      (res_s2t['chrf']      + res_t2s['chrf'])      / 2,
        'chrfpp':    (res_s2t['chrfpp']    + res_t2s['chrfpp'])    / 2,
    },
    'n_test': len(test_data),
}
print(json.dumps(summary, indent=2, ensure_ascii=False))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'test_metrics.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False))

with (OUTPUT_DIR / 'test_predictions.jsonl').open('w') as fh:
    for direction, res in (('es->bri', res_s2t), ('bri->es', res_t2s)):
        for pred, ref in zip(res['predictions'], res['references']):
            fh.write(json.dumps({'direction': direction, 'prediction': pred, 'reference': ref}, ensure_ascii=False) + '\n')
print(f'Guardado: {OUTPUT_DIR}/test_metrics.json y {OUTPUT_DIR}/test_predictions.jsonl')

## 7. Curvas de entrenamiento

Gráficas aplicables a este proyecto: `val loss`, `spBLEU` y `chrF++`. ROC, matriz de confusión y precisión-recall no aplican directamente porque el problema es generación de secuencias, no clasificación binaria/multiclase.

In [ ]:
import json
import matplotlib.pyplot as plt

hist = json.loads((OUTPUT_DIR / 'metrics.json').read_text())
steps = hist['steps']

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(steps, hist['src2tgt']['loss'], label='es→bri')
axes[0].plot(steps, hist['tgt2src']['loss'], label='bri→es')
axes[0].plot(steps, hist['avg']['loss'], label='avg', linestyle='--')
axes[0].set_title('Val loss'); axes[0].set_xlabel('step'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(steps, hist['src2tgt']['spbleu'], label='es→bri')
axes[1].plot(steps, hist['tgt2src']['spbleu'], label='bri→es')
axes[1].plot(steps, hist['avg']['spbleu'], label='avg', linestyle='--')
axes[1].set_title('spBLEU'); axes[1].set_xlabel('step'); axes[1].legend(); axes[1].grid(True)

axes[2].plot(steps, hist['src2tgt']['chrfpp'], label='es→bri')
axes[2].plot(steps, hist['tgt2src']['chrfpp'], label='bri→es')
axes[2].plot(steps, hist['avg']['chrfpp'], label='avg', linestyle='--')
axes[2].set_title('chrF++'); axes[2].set_xlabel('step'); axes[2].legend(); axes[2].grid(True)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=140)
plt.show()

## 8. Tabla comparativa del Avance 5

Esta celda consolida los modelos previos del README y agrega la corrida 3.3B cuando `test_metrics.json` existe. La tabla queda ordenada por `chrF++` promedio, la métrica primaria de selección para este proyecto.

In [ ]:
import json
from pathlib import Path

runs = [
    ('NLLB orig · 600M · 3ep lr5e-4', Path('outputs/test_metrics.json'), 'T4', '30-45 min reportado'),
    ('NLLB H100 · 600M · 8ep lr2e-4', Path('outputs_nllb_h100/test_metrics.json'), 'H100', 'no registrado'),
    ('M2M-100 · 418M · 3ep lr5e-4', Path('outputs_m2m100/test_metrics.json'), 'H100', 'no registrado'),
    ('NLLB · 3.3B · 3ep lr5e-4', OUTPUT_DIR / 'test_metrics.json', 'Colab A100', 'registrado en metrics.json'),
]

rows = []
for name, path, hw, train_time in runs:
    if not path.exists():
        continue
    metrics = json.loads(path.read_text())
    avg = metrics['avg']
    rows.append({
        'Modelo': name,
        'Hardware': hw,
        'Tiempo': train_time,
        'eval_loss': avg['eval_loss'],
        'spBLEU': avg['spbleu'],
        'chrF': avg['chrf'],
        'chrF++': avg['chrfpp'],
    })

rows.sort(key=lambda row: row['chrF++'], reverse=True)
headers = ['Modelo', 'Hardware', 'Tiempo', 'eval_loss', 'spBLEU', 'chrF', 'chrF++']
print('| ' + ' | '.join(headers) + ' |')
print('| ' + ' | '.join(['---', '---', '---', '---:', '---:', '---:', '---:']) + ' |')
for row in rows:
    print('| {Modelo} | {Hardware} | {Tiempo} | {eval_loss:.3f} | {spBLEU:.2f} | {chrF:.2f} | {chrF++:.2f} |'.format(**row))

## 9. Detener heartbeat

In [ ]:
_HEARTBEAT_STOP = True
print('heartbeat detenido')